# CRISPYx Tutorial

This tutorial walks through the Scanpy-style workflow provided by ``crispyx``.

## Prerequisites

Install the project in editable mode (with the optional ``test`` extras) to make the package importable and provide AnnData, NumPy, pandas, and SciPy:

```bash
pip install -e .[test]
```

## Prepare a demo dataset

The repository includes a small synthetic perturbation screen stored at ``data/demo_benchmark.h5ad``. Run ``python benchmarking/generate_demo_dataset.py`` to regenerate it if necessary.

In [1]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))
DATA_PATH = ROOT / 'data' / 'demo_benchmark.h5ad'
OUTPUT_DIR = ROOT / 'notebooks' / 'tutorial_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import crispyx as cx
from benchmarking.generate_demo_dataset import write_demo_dataset

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    write_demo_dataset(DATA_PATH)

## Inspect the on-disk AnnData

``cx.read_h5ad_ondisk`` returns a read-only :class:`cx.AnnData` wrapper. The ``obs`` and ``var`` accessors expose ``head`` and ``load`` helpers so you can preview metadata without materialising the entire object.

In [2]:
adata_ro = cx.read_h5ad_ondisk(DATA_PATH)
adata_ro

AnnData object with n_obs × n_vars = 400 × 100 backed at '/home/jinhongd/Streamlining-CRISPR-Screen-Analysis/data/demo_benchmark.h5ad'
    obs: 'perturbation', 'celltype'
    var: 'gene_symbols'
    uns: 'description'
First obs rows:
         perturbation celltype
cell                          
cell_000    perturb_3  NK cell
cell_001         ctrl   T cell
cell_002         ctrl   B cell
cell_003         ctrl  NK cell
cell_004    perturb_4  NK cell
First var rows:
         gene_symbols
gene                 
gene_000         G000
gene_001         G001
gene_002         G002
gene_003         G003
gene_004         G004


AnnData(path=/home/jinhongd/Streamlining-CRISPR-Screen-Analysis/data/demo_benchmark.h5ad, mode='r')

In [3]:
adata_ro.obs.head()

,perturbation,celltype
cell,,
cell_000,perturb_3,NK cell
cell_001,ctrl,T cell
cell_002,ctrl,B cell
cell_003,ctrl,NK cell
cell_004,perturb_4,NK cell


In [4]:
adata_ro.var.head()

,gene_symbols
gene,
gene_000,G000
gene_001,G001
gene_002,G002
gene_003,G003
gene_004,G004


## Quality control with ``cx.pp``

The preprocessing namespace mirrors Scanpy's API. Each function returns a new ``cx.AnnData`` object backed by an ``.h5ad`` file.

In [ ]:
qc_result = cx.quality_control_summary(
    adata_ro,
    perturbation_column='perturbation',
    min_genes=5,
    min_cells_per_perturbation=5,
    output_dir=OUTPUT_DIR,
    data_name='tutorial',
)
qc_preview = pd.DataFrame(
    {
        'perturbation': adata_ro.obs['perturbation'],
        'qc_pass': qc_result.cell_mask,
    }
)
adata_ro = qc_result.filtered
qc_preview.head()


,perturbation,qc_pass
cell,,
cell_000,perturb_3,True
cell_001,ctrl,True
cell_002,ctrl,True
cell_003,ctrl,True
cell_004,perturb_4,True


## Pseudobulk aggregation with ``cx.pb``

Aggregate cells per perturbation on disk. The returned object can be inspected lazily, similar to the original dataset.

In [ ]:
adata_pb_ro = cx.pb.average_log_expression(
    adata_ro,
    perturbation_column='perturbation',
    output_dir=OUTPUT_DIR,
    data_name='tutorial',
)
adata_pb_ro.var.head()

""
gene
gene_000
gene_001
gene_002
gene_003
gene_004


## Differential expression with ``scr.tl``

Run a Wilcoxon test that mirrors ``scanpy.tl.rank_genes_groups``. ``scr.AnnData.uns`` entries also provide ``preview`` and ``load`` helpers.

In [ ]:
adata_ro = cx.tl.rank_genes_groups(
    adata_ro,
    perturbation_column='perturbation',
    method='wilcoxon',
    output_dir=OUTPUT_DIR,
    data_name='tutorial',
)
adata_ro.uns['rank_genes_groups'].preview()

/home/jinhongd/Streamlining-CRISPR-Screen-Analysis/src/crispyx/data.py:116: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.
  return list(self._parent.backed.uns_keys())


{'auc': array([(0.4994709 , 0.49968774, 0.49782313, 0.49876543, 0.49914966),
        (0.49647266, 0.4950039 , 0.49959184, 0.49823633, 0.49710884),
        (0.49929453, 0.49875098, 0.4970068 , 0.49700176, 0.49761905),
        (0.49726631, 0.49968774, 0.49659864, 0.49559083, 0.49438776),
        (0.49497354, 0.49703357, 0.49591837, 0.49541446, 0.49285714)],
       dtype=[('perturb_3', '<f8'), ('perturb_4', '<f8'), ('perturb_5', '<f8'), ('perturb_2', '<f8'), ('perturb_1', '<f8')]),
 'full': {'auc': array([[-0.11402116, -0.05828924, -0.08121693, -0.08095238, -0.13209877,
          -0.12107584, -0.03783069, -0.08783069, -0.07583774, -0.08342152,
          -0.12469136, -0.11349206, -0.09021164, -0.11684303, -0.07257496,
          -0.08800705, -0.05502646, -0.15970018, -0.03333333, -0.07425044,
          -0.12795414,  0.49497354,  0.49929453,  0.49726631,  0.4994709 ,
           0.49647266, -0.05176367, -0.07425044, -0.05890653, -0.13386243,
          -0.1468254 ,  0.01067019, -0.11604938, -0

Use ``load`` to materialise the complete differential expression tables for downstream analysis.

In [8]:
wilcoxon = adata_ro.uns['rank_genes_groups'].load()
wilcoxon_full = adata_ro.uns['rank_genes_groups'].load()['full']
groups = list(wilcoxon['names'].dtype.names)
group = groups[0]
group_index = groups.index(group)
top_genes = wilcoxon['names'][group]
pd.DataFrame({
    'gene': top_genes,
    'logfoldchange': wilcoxon_full['logfoldchanges'][group_index][: top_genes.size],
    'pval_adj': wilcoxon_full['pvals_adj'][group_index][: top_genes.size],
}).head()


/home/jinhongd/Streamlining-CRISPR-Screen-Analysis/src/crispyx/data.py:116: FutureWarning: Use uns (e.g. `k in adata.uns` or `sorted(adata.uns)`) instead of AnnData.uns_keys, AnnData.uns_keys is deprecated and will be removed in the future.
  return list(self._parent.backed.uns_keys())


,gene,logfoldchange,pval_adj
0,gene_024,-0.276430,0.040696
1,gene_025,0.170924,0.258447
2,gene_022,-0.203878,0.137966
3,gene_023,-0.094676,0.138738
4,gene_021,-0.695081,0.027262


The ``cx.AnnData`` handles close themselves when their Python objects go out of scope, so no explicit cleanup is required.